# Caso B · 03 Features para forecasting horario

> _Tutorial · Caso de uso: **B — Forecast consumo 24h** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Construir el dataset de features (oro) para entrenar modelos de predicción del consumo eléctrico a 24h. Generar lags, rolling, calendario lectivo y variables exógenas (T_ext, GHI).


## 2. Qué se aprende

- Lag features (1h, 24h, 168h).
- Rolling means (7d, 24h).
- Codificación cíclica de hora/día.
- Variables exógenas y posibles fugas de información.


## 3. Contexto del caso de uso

Los modelos SARIMA / XGBoost / LSTM esperan formatos distintos. Aquí construimos un **dataset largo** con todas las features y un esquema tabular en cuatro columnas (X, y, time, asset).


## 4. Relación con CENTINELA+

Las mismas features se calculan en producción cada hora antes de invocar el modelo. La función `make_features(df)` debe ser pura para reproducibilidad.


## 5. Relación con Medallion

**Capa oro** específica del Caso B.


## 6. Datos de entrada

Mock `bdg2_education_subset_mock.csv`. En modo online, se leería de Influx.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

Las features no se publican como `captia_point`; viven en pandas / Parquet.


## 9. Carga de datos o mock

Cargamos y pivotamos a un DataFrame ancho.


In [2]:
csv_path = ROOT / "notebooks" / "_data" / "bdg2_education_subset_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"])
df = df[df.building_id == df.building_id.unique()[0]].sort_values("timestamp").set_index("timestamp")
df.head()


,building_id,power_kw,t_outdoor,ghi
timestamp,,,,
2024-01-01 00:00:00+00:00,bdg2_bldg_00,17.63,-2.16,0.0
2024-01-01 01:00:00+00:00,bdg2_bldg_00,8.78,-0.06,0.0
2024-01-01 02:00:00+00:00,bdg2_bldg_00,5.09,-4.22,0.0
2024-01-01 03:00:00+00:00,bdg2_bldg_00,13.14,-3.75,0.0
2024-01-01 04:00:00+00:00,bdg2_bldg_00,14.35,-0.40,0.0


## 10. Exploración paso a paso

Verificamos cobertura horaria sin huecos.


In [3]:
gaps = df.index.to_series().diff().dt.total_seconds().dropna()
print("Mediana entre puntos (s):", gaps.median(), "  Máximo:", gaps.max())


Mediana entre puntos (s): 3600.0   Máximo: 3600.0


## 11. Transformación bronce → plata

Construimos features (no escribimos a InfluxDB; el oro vive como Parquet).


In [4]:
def make_features(d):
    out = pd.DataFrame(index=d.index)
    out["y"] = d["power_kw"]
    out["t_outdoor"] = d["t_outdoor"]
    out["ghi"] = d["ghi"]
    # Cyclical
    h = d.index.hour
    out["hour_sin"] = np.sin(2 * np.pi * h / 24)
    out["hour_cos"] = np.cos(2 * np.pi * h / 24)
    out["dow"] = d.index.dayofweek
    out["is_weekend"] = (out["dow"] >= 5).astype(int)
    # Lags
    out["lag_1h"] = d["power_kw"].shift(1)
    out["lag_24h"] = d["power_kw"].shift(24)
    out["lag_168h"] = d["power_kw"].shift(168)
    # Rolling
    out["roll_24h_mean"] = d["power_kw"].shift(1).rolling(24).mean()
    out["roll_168h_mean"] = d["power_kw"].shift(1).rolling(168).mean()
    return out.dropna()

X = make_features(df)
X.head()


,y,t_outdoor,ghi,hour_sin,hour_cos,dow,is_weekend,lag_1h,lag_24h,lag_168h,roll_24h_mean,roll_168h_mean
timestamp,,,,,,,,,,,,
2024-01-08 00:00:00+00:00,10.45,-1.12,0.0,0.000000,1.000000,0,0,12.82,15.13,17.63,11.071667,14.481726
2024-01-08 01:00:00+00:00,10.69,4.16,0.0,0.258819,0.965926,0,0,10.45,12.20,8.78,10.876667,14.438988
2024-01-08 02:00:00+00:00,8.42,0.46,0.0,0.500000,0.866025,0,0,10.69,15.28,5.09,10.813750,14.450357
2024-01-08 03:00:00+00:00,10.41,1.04,0.0,0.707107,0.707107,0,0,8.42,14.45,13.14,10.527917,14.470179
2024-01-08 04:00:00+00:00,7.69,-0.67,0.0,0.866025,0.500000,0,0,10.41,8.19,14.35,10.359583,14.453929


## 12. Construcción de capa oro

Persistimos a Parquet (oro local).


In [5]:
out_dir = ROOT / "output" / "case_B"
out_dir.mkdir(parents=True, exist_ok=True)
parquet_path = out_dir / "features_b0.parquet"
X.to_parquet(parquet_path)
print(f"Wrote {parquet_path.relative_to(ROOT)} ({len(X)} filas)")


Wrote output\case_B\features_b0.parquet (8472 filas)


## 13. Visualizaciones explicativas

Mostramos las correlaciones más altas con `y`.


In [6]:
correls = X.drop(columns=["y"]).apply(lambda c: c.corr(X["y"])).sort_values()
correls.plot.barh(figsize=(7, 4), color="#FF5722")
plt.title("Correlación de features con y (power_kw)")
plt.tight_layout()


## 14. Validaciones

Sin NaN tras `dropna`, lags correctos.


In [7]:
assert X.isna().sum().sum() == 0
assert X["lag_24h"].iloc[0] == X["y"].iloc[0] - (X["y"].iloc[0] - X["lag_24h"].iloc[0])  # tautología, pero comprueba shape
print("Features shape:", X.shape)


Features shape: (8472, 12)


## 15. Errores comunes

1. **Leakage temporal**: usar `.rolling()` sin `shift(1)` mezcla pasado y futuro.
2. **Codificación de `dow` no cíclica**: lunes y domingo aparecerán muy lejos en el espacio de features. Usar sen/cos.
3. **Imputar NaN con la media**: para series temporales mejor `ffill` o drop.


## 16. Ejercicios propuestos

1. Añade `lag_24h_diff = y - lag_24h` y mide su correlación.
2. Crea una feature de calendario lectivo (Comunidad Valenciana).
3. Compara MAE de un modelo con y sin features cíclicas.


## 17. Cómo se reutiliza con datos reales

`make_features(df)` es pura: misma firma, distinto origen. La única adaptación es la columna `power_kw` ↔ `power_01` en CENTINELA+.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `02_case_B_energy_forecasting/04_baseline_sarima_xgboost_lstm.ipynb`.
- Documento web del caso: `docs/use-cases/case-b-energy-forecasting.md`.


## 19. Marco teórico (nivel doctoral)

### Modelo SARIMA

$\text{SARIMA}(p,d,q)(P,D,Q)_s$ con período estacional $s$:

$$
\Phi_P(B^s)\,\phi_p(B)\,(1-B)^d\,(1-B^s)^D\,y_t = \Theta_Q(B^s)\,\theta_q(B)\,\varepsilon_t
$$

con $B$ operador retardo, $\varepsilon_t \sim \mathcal{N}(0, \sigma^2)$. Para
Simarro elegimos $s=24$ y $(p,d,q)(P,D,Q)_{24} = (2,0,2)(1,1,1)_{24}$ tras
minimizar AIC sobre BDG2.

### XGBoost regularizado (Chen & Guestrin 2016)

$$
\hat{y}_t = \sum_{k=1}^{K} f_k(\mathbf{x}_t), \quad f_k \in \mathcal{F}
$$

con función objetivo

$$
\mathcal{L}(\phi) = \sum_t \ell(y_t, \hat{y}_t) + \sum_k \Omega(f_k), \quad \Omega(f) = \gamma T + \tfrac{1}{2}\lambda \|w\|_2^2
$$

### LSTM (Hochreiter & Schmidhuber 1997)

$$
\begin{aligned}
f_t &= \sigma(W_f [h_{t-1}, x_t] + b_f) \\
i_t &= \sigma(W_i [h_{t-1}, x_t] + b_i) \\
\tilde{c}_t &= \tanh(W_c [h_{t-1}, x_t] + b_c) \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \\
o_t &= \sigma(W_o [h_{t-1}, x_t] + b_o) \\
h_t &= o_t \odot \tanh(c_t)
\end{aligned}
$$

### Métricas

$$
\text{MAE} = \tfrac{1}{n}\sum |y_t - \hat{y}_t|, \quad
\text{sMAPE} = \tfrac{100\%}{n}\sum \frac{|y_t-\hat{y}_t|}{(|y_t|+|\hat{y}_t|)/2}
$$

Objetivos Simarro: $\text{MAE} \leq 0.15$ kWh, $\text{sMAPE} \leq 12\%$.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

Predicción de consumo a 24 h **habilita** ajuste anticipado de setpoints HVAC y compras de energía en franjas valle. Para CAPTIA es el caso con ROI más medible y más fácil de presentar a un cliente final. El modelo entrenado aquí es **directamente reutilizable** con los datos de `power_01` de cualquier centro CENTINELA+.

### ROI estimado

| Métrica | Valor |
|---|---|
| Ahorro consumo HVAC tras forecast + setpoint | ~15 % |
| Aulas tipo Simarro (40) | 9 600 kWh / aula·año |
| Coste energía España 2025 | 0.14 €/kWh |
| **Ahorro centro:** $40 \cdot 9\,600 \cdot 0.14 \cdot 0.15$ | **8 064 €/año** |
| Coste implantación | ~3 000 € one-time |
| **Payback** | **~5 meses** |

### Riesgos y mitigaciones

- Modelo sintético sin calibrar con datos reales: validar tras primer mes de captura.
- Drift estacional: re-entrenar trimestralmente.


## 21. Bibliografía y referencias

- Box, G. E. P., Jenkins, G. M., Reinsel, G. C. & Ljung, G. M. (2015). *Time Series Analysis: Forecasting and Control* (5ª ed.). Wiley.
- Chen, T. & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System*. KDD '16.
- Hochreiter, S. & Schmidhuber, J. (1997). *Long Short-Term Memory*. Neural Computation 9(8).
- Miller, C. et al. (2020). *The Building Data Genome 2 (BDG2) data-set*. Scientific Data.
- ASHRAE (2022). *ASHRAE 90.1-2022 — Energy Standard for Buildings*.
